# Notebook 2 — Synthetic Dataset Generation

**Purpose:** Generate the two synthetic drift datasets used in this study
by injecting controlled distributional shifts into the Electricity dataset structure.

**Why synthetic datasets?** The real Electricity dataset exhibits naturally
occurring drift that cannot be controlled or isolated. Synthetic datasets allow
systematic evaluation of retraining strategies under known, reproducible drift
conditions with precise control over drift type, timing, and severity.

**Datasets generated:**
| File | Drift Type | Mechanism |
|------|------------|----------|
| `gradual_concept.xlsx` | Gradual covariate + concept drift | P(X) and P(Y\|X) shift progressively across batches |
| `abrupt_concept.xlsx` | Abrupt covariate + concept drift | P(X) and P(Y\|X) shift as step-function at fixed batch indices |

**Output location:** `ready_to_use_datasets/` (same folder as `electricity_clean.xlsx`)

> **Prerequisite:** Run Notebook 1 first to generate `electricity_clean.xlsx`.
> The synthetic datasets share the same feature names, batch structure,
> and row count (45,000 rows = 90 batches × 500 rows).

---
| Cell | Content |
|------|---------|
| 1 | Mount Google Drive & configure paths |
| 2 | Imports & constants |
| 3 | Temporal structure helper |
| 4 | Gradual Covariate + Concept Drift generator |
| 5 | Abrupt Covariate + Concept Drift generator |
| 6 | Generate & save both datasets |
| 7 | Validation — verify outputs |

## Cell 1 — Mount Google Drive & Configure Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── PATH CONFIGURATION ──────────────────────────────────────────
# Output folder — same location as electricity_clean.xlsx from Notebook 1
DATA_PATH = (
    "/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/"
)

OUTPUT_FILES = {
    'gradual_concept': os.path.join(DATA_PATH, 'gradual_concept.xlsx'),
    'abrupt_concept' : os.path.join(DATA_PATH, 'abrupt_concept.xlsx'),
}

print('Output paths:')
for name, path in OUTPUT_FILES.items():
    print(f'  {name}: {path}')


Mounted at /content/drive
Output paths:
  gradual_concept: /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/gradual_concept.xlsx
  abrupt_concept: /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/abrupt_concept.xlsx


## Cell 2 — Imports & Constants

All synthetic datasets use the same feature names and structure as the
real Electricity dataset to ensure compatibility with Notebooks 3–5.

In [2]:
import numpy as np
import pandas as pd

# Must match Notebook 1 configuration exactly
BATCH_SIZE          = 500
TOTAL_SAMPLES       = 45_000   # 90 batches × 500 rows
INITIAL_TRAIN_RATIO = 0.20    # first 20% = baseline (no drift injected)
TARGET              = 'class' # binary: 0 = DOWN, 1 = UP
SEED                = 42

# Derived
N_BATCHES   = TOTAL_SAMPLES // BATCH_SIZE  # 90
BASELINE_B  = int(N_BATCHES * INITIAL_TRAIN_RATIO)  # 18 batches

print(f'Total batches  : {N_BATCHES}')
print(f'Baseline batches: {BASELINE_B}  (no drift, batches 0–{BASELINE_B-1})')
print(f'Drift batches   : {N_BATCHES - BASELINE_B}  (drift active, batches {BASELINE_B}–{N_BATCHES-1})')


Total batches  : 90
Baseline batches: 18  (no drift, batches 0–17)
Drift batches   : 72  (drift active, batches 18–89)


## Cell 3 — Temporal Structure Helper

`add_temporal_structure()` adds the `date`, `day_clean`, and `period` columns
that mirror the real Electricity dataset. These columns are required by the
simulation engine in Notebook 4 but are not used in the drift injection logic.

In [3]:
def add_temporal_structure(df: pd.DataFrame) -> pd.DataFrame:
    """Add date/time columns mirroring the real Electricity dataset structure."""
    df = df.copy()
    df['datetime'] = pd.date_range(start='1996-01-01', periods=len(df), freq='30min')
    df['date']      = df['datetime'].dt.date
    df['day_clean'] = df['datetime'].dt.dayofweek
    df['period']    = (
        df['datetime'].dt.hour * 60 + df['datetime'].dt.minute
    ) / (24 * 60)
    return df.drop(columns=['datetime'])

print('✓ add_temporal_structure() defined.')


✓ add_temporal_structure() defined.


## Cell 4 — Gradual Covariate + Concept Drift Generator

**Drift mechanism:**
- **Covariate shift P(X):** Feature means shift linearly as a function of
  batch progress (`shift = progress × 0.1`). Drift begins at batch 18
  (after the baseline window) and increases monotonically to batch 89.
- **Concept drift P(Y|X):** The decision boundary gradually transitions
  from `score = nswprice − vicprice` (regime 0) to
  `score = 0.5·nswprice − 0.8·vicprice + 0.3·transfer` (regime 1).
  The interpolation weight equals `progress`, so the boundary shifts
  continuously rather than abruptly.

**Expected PSI pattern:** Monotonically rising from ~0 at batch 0 to ~0.95 by batch 89.
**Expected accuracy pattern:** Steady decline from ~0.97 to ~0.79 under a static model.

In [4]:
def generate_gradual_concept(batch_size: int = BATCH_SIZE, seed: int = SEED) -> pd.DataFrame:
    """
    Generate gradual covariate + concept drift dataset.
    Both P(X) and P(Y|X) shift progressively starting from batch BASELINE_B.
    """
    np.random.seed(seed)
    data = []

    for b in range(N_BATCHES):
        # progress = 0 during baseline, ramps 0→1 across drift window
        progress = 0.0 if b < BASELINE_B else \
                   (b - BASELINE_B) / (N_BATCHES - BASELINE_B)
        shift    = progress * 0.1

        # ── Covariate shift: feature means drift linearly ────────────────
        nswprice  = np.random.normal(0.5 + shift, 0.1, batch_size)
        vicprice  = np.random.normal(0.4 + shift, 0.1, batch_size)
        nswdemand = np.random.normal(0.5 - shift, 0.1, batch_size)
        vicdemand = np.random.normal(0.4 - shift, 0.1, batch_size)
        transfer  = np.random.normal(0.3 + shift, 0.05, batch_size)
        noise     = np.random.normal(0, 0.02, batch_size)

        # ── Concept drift: decision boundary interpolates between regimes ─
        score = (
            (1 - progress) * (nswprice - vicprice) +
            progress * (0.5 * nswprice - 0.8 * vicprice + 0.3 * transfer)
        ) + noise
        y = (score > 0).astype(int)

        data.append(pd.DataFrame({
            'nswprice': nswprice, 'nswdemand': nswdemand,
            'vicprice': vicprice, 'vicdemand': vicdemand,
            'transfer': transfer, TARGET: y, 'batch_id': b
        }))

    df = pd.concat(data).reset_index(drop=True)
    return add_temporal_structure(df)

print('✓ generate_gradual_concept() defined.')


✓ generate_gradual_concept() defined.


## Cell 5 — Abrupt Covariate + Concept Drift Generator

**Drift mechanism:**
- **Covariate shift P(X):** Feature means shift as a step function cycling
  through 3 regimes every 12 batches: shift = 0.0 → 0.08 → 0.15 → repeat.
- **Concept drift P(Y|X):** The decision boundary also changes abruptly
  per regime:
  - Regime 0: `score = nswprice − vicprice` (original boundary)
  - Regime 1: `score = 0.5·nswprice − 0.8·vicprice + 0.3·transfer`
  - Regime 2: `score = −0.3·nswprice + 0.6·vicprice + 0.5·transfer` (inverted)

**Expected PSI pattern:** Step-function blocks alternating between PSI ≈ 1.0
(during shifted regimes) and PSI ≈ 0 (when distribution returns to baseline).
**Expected accuracy pattern:** Sharp drops at each regime transition at
batch indices 18, 30, 42, 54, 66 (every 12 batches after baseline).

In [5]:
def generate_abrupt_concept(batch_size: int = BATCH_SIZE, seed: int = SEED) -> pd.DataFrame:
    """
    Generate abrupt covariate + concept drift dataset.
    Both P(X) and P(Y|X) shift as step functions cycling every 12 batches.
    """
    np.random.seed(seed)

    REGIME_SHIFTS     = [0.0, 0.08, 0.15]   # covariate shift magnitudes
    REGIME_INTERVAL   = 12                   # batches per regime cycle
    data = []

    for b in range(N_BATCHES):
        # Regime selection: 0 during baseline, cycles 0→1→2→0... during drift
        if b < BASELINE_B:
            regime = 0
        else:
            cycle  = (b - BASELINE_B) // REGIME_INTERVAL
            regime = cycle % len(REGIME_SHIFTS)

        shift = REGIME_SHIFTS[regime]

        # ── Covariate shift: abrupt step in feature means ────────────────
        nswprice  = np.random.normal(0.5 + shift, 0.1, batch_size)
        vicprice  = np.random.normal(0.4 + shift, 0.1, batch_size)
        nswdemand = np.random.normal(0.5 - shift, 0.1, batch_size)
        vicdemand = np.random.normal(0.4 - shift, 0.1, batch_size)
        transfer  = np.random.normal(0.3 + shift, 0.05, batch_size)
        noise     = np.random.normal(0, 0.02, batch_size)

        # ── Concept drift: decision boundary flips per regime ────────────
        if regime == 0:
            score = nswprice - vicprice + noise
        elif regime == 1:
            score = 0.5 * nswprice - 0.8 * vicprice + 0.3 * transfer + noise
        else:  # regime == 2
            score = -0.3 * nswprice + 0.6 * vicprice + 0.5 * transfer + noise

        y = (score > 0).astype(int)

        data.append(pd.DataFrame({
            'nswprice': nswprice, 'nswdemand': nswdemand,
            'vicprice': vicprice, 'vicdemand': vicdemand,
            'transfer': transfer, TARGET: y, 'batch_id': b
        }))

    df = pd.concat(data).reset_index(drop=True)
    return add_temporal_structure(df)

print('✓ generate_abrupt_concept() defined.')


✓ generate_abrupt_concept() defined.


## Cell 6 — Generate & Save Both Datasets

Generates both datasets and saves them to `ready_to_use_datasets/`.
If a file already exists it is skipped — delete the file manually
to regenerate with different parameters.

> **Estimated runtime:** ~1–2 minutes per dataset.

In [6]:
GENERATE_MAP = {
    'gradual_concept': generate_gradual_concept,
    'abrupt_concept' : generate_abrupt_concept,
}

for name, gen_fn in GENERATE_MAP.items():
    fpath = OUTPUT_FILES[name]
    if os.path.exists(fpath):
        print(f'⏭️  {name} already exists — skipping. (Delete file to regenerate.)')
    else:
        print(f'▶️  Generating {name} ...')
        df = gen_fn()
        df.to_excel(fpath, index=False)
        print(f'   ✓ Saved → {fpath}')
        print(f'   Rows: {len(df):,}  Batches: {df["batch_id"].nunique()}')
        print(f'   Class balance: {df[TARGET].mean():.3f} (fraction UP)')

print('\n✅ Dataset generation complete.')


▶️  Generating gradual_concept ...
   ✓ Saved → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/gradual_concept.xlsx
   Rows: 45,000  Batches: 90
   Class balance: 0.697 (fraction UP)
▶️  Generating abrupt_concept ...
   ✓ Saved → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/abrupt_concept.xlsx
   Rows: 45,000  Batches: 90
   Class balance: 0.774 (fraction UP)

✅ Dataset generation complete.


## Cell 7 — Validation

Verifies that both generated files match the expected structure
and are consistent with `electricity_clean.xlsx` from Notebook 1.

In [7]:
import os

EXPECTED_ROWS = TOTAL_SAMPLES
EXPECTED_COLS = {'nswprice', 'nswdemand', 'vicprice', 'vicdemand',
                 'transfer', 'class', 'batch_id', 'date', 'day_clean', 'period'}

print('=== Dataset Validation ===')
all_ok = True
for name, fpath in OUTPUT_FILES.items():
    print(f'\n  {name}')
    df = pd.read_excel(fpath)

    row_ok  = len(df) == EXPECTED_ROWS
    col_ok  = EXPECTED_COLS.issubset(set(df.columns))
    bat_ok  = df['batch_id'].nunique() == N_BATCHES
    nan_ok  = df.isnull().sum().sum() == 0
    cls_ok  = set(df['class'].unique()).issubset({0, 1})

    print(f'    Rows     : {len(df):,}  ["✓" if row_ok else "✗"]')
    print(f'    Columns  : {list(df.columns)}  ["✓" if col_ok else "✗"]')
    print(f'    Batches  : {df["batch_id"].nunique()}  ["✓" if bat_ok else "✗"]')
    print(f'    No NaNs  : ["✓" if nan_ok else "✗"]')
    print(f'    Class ∈ {{0,1}}: ["✓" if cls_ok else "✗"]')
    print(f'    Class balance: {df["class"].mean():.3f}')

    if not all([row_ok, col_ok, bat_ok, nan_ok, cls_ok]):
        all_ok = False

print(f'\n{"✅ All datasets valid." if all_ok else "❌ Issues found — check above."}')


=== Dataset Validation ===

  gradual_concept
    Rows     : 45,000  ["✓" if row_ok else "✗"]
    Columns  : ['nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer', 'class', 'batch_id', 'date', 'day_clean', 'period']  ["✓" if col_ok else "✗"]
    Batches  : 90  ["✓" if bat_ok else "✗"]
    No NaNs  : ["✓" if nan_ok else "✗"]
    Class ∈ {0,1}: ["✓" if cls_ok else "✗"]
    Class balance: 0.697

  abrupt_concept
    Rows     : 45,000  ["✓" if row_ok else "✗"]
    Columns  : ['nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer', 'class', 'batch_id', 'date', 'day_clean', 'period']  ["✓" if col_ok else "✗"]
    Batches  : 90  ["✓" if bat_ok else "✗"]
    No NaNs  : ["✓" if nan_ok else "✗"]
    Class ∈ {0,1}: ["✓" if cls_ok else "✗"]
    Class balance: 0.774

✅ All datasets valid.
